# Debug Query Workbench

Ask a question and inspect retrieval + LLM debug details directly from the notebook.

This mirrors the API debug page sections: question, rewrite, sources, hybrid ranking, request payload, and response.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import IPython.display as ipd


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "core" / "llmapi.py").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing core/llmapi.py")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. LLM calls will fail until it is configured.")

from core import llmapi  # noqa: E402

print(f"Repo root: {REPO_ROOT}")
print(f"LLM model: {llmapi.LLM_MODEL}")

Repo root: C:\python\hierag
LLM model: gpt-5.2


In [2]:
def _to_table(rows, columns):
    try:
        import pandas as pd

        if not rows:
            return pd.DataFrame(columns=columns)
        return pd.DataFrame(rows)[columns]
    except Exception:
        if not rows:
            return []
        return [{k: row.get(k) for k in columns} for row in rows]


def _render_debug_sections(debug_payload: dict):
    debug_payload = debug_payload or {}
    retrieval = debug_payload.get("retrieval") or {}
    ranked_chunks = retrieval.get("ranked_chunks") or []
    sources = debug_payload.get("sources") or []
    query_effective = debug_payload.get("query_effective") or debug_payload.get("query") or ""
    query_rewritten = debug_payload.get("query_rewritten") or ""
    query_rewrite = debug_payload.get("query_rewrite") or {}
    llm_request = debug_payload.get("llm_request") or {}
    llm_response_text = debug_payload.get("llm_response_text") or ""

    rewrite_used = bool(query_rewrite.get("used"))
    rewrite_rows = [
        {"field": "status", "value": "rewritten" if rewrite_used else "not rewritten"},
        {"field": "reason", "value": query_rewrite.get("reason") or ("used" if rewrite_used else "not_available")},
        {"field": "model", "value": query_rewrite.get("model") or "-"},
        {"field": "history_turns", "value": query_rewrite.get("history_turns", "-")},
    ]
    if query_rewrite.get("error"):
        rewrite_rows.append({"field": "error", "value": query_rewrite.get("error")})

    source_columns = [
        "score",
        "from_vector",
        "from_bm25",
        "vector_score_raw",
        "bm25_score_raw",
        "extract_id",
        "url",
        "last_scraped",
    ]

    ranking_columns = [
        "rank",
        "score",
        "from_vector",
        "from_bm25",
        "vector_score_norm",
        "bm25_score_norm",
        "vector_score_raw",
        "bm25_score_raw",
        "chunk_id",
        "url",
    ]

    ipd.display(ipd.Markdown("## Question"))
    print(debug_payload.get("query") or "-")

    ipd.display(ipd.Markdown("## Query Rewrite"))
    ipd.display(_to_table(rewrite_rows, ["field", "value"]))
    print("\nEffective query used for retrieval:")
    print(query_effective or "-")
    if query_rewritten:
        print("\nRewritten standalone query:")
        print(query_rewritten)

    candidate_counts = retrieval.get("candidate_counts") or {}
    ipd.display(ipd.Markdown("## Sources"))
    ipd.display(_to_table(sources, source_columns))

    ipd.display(ipd.Markdown("## Hybrid Ranking"))
    print(
        "vector candidates: "
        f"{candidate_counts.get('vector', '-')} | "
        "bm25 candidates: "
        f"{candidate_counts.get('bm25', '-')} | "
        "merged: "
        f"{candidate_counts.get('merged', '-')}"
    )
    ipd.display(_to_table(ranked_chunks, ranking_columns))

    ipd.display(ipd.Markdown("## LLM Request Payload"))
    print("System:")
    print(llm_request.get("system_text") or "-")
    print("\nUser (question + context):")
    print(llm_request.get("user_text") or "-")

    ipd.display(ipd.Markdown("## LLM Response"))
    print(llm_response_text or "-")


def ask_with_debug(query: str, *, top_k: int = 10, max_extracts: int = 6, history: list[dict] | None = None):
    query = (query or "").strip()
    if not query:
        raise ValueError("query must be non-empty")

    events = []
    answer_chunks = []
    debug_payload = {}

    for event in llmapi.stream_answer_with_context(
        query,
        top_k=top_k,
        max_extracts=max_extracts,
        history=history,
    ):
        events.append(event)
        etype = event.get("type")
        if etype == "delta":
            text = event.get("text", "")
            answer_chunks.append(text)
            print(text, end="", flush=True)
        elif etype == "debug":
            debug_payload = event.get("debug") or {}
        elif etype == "error":
            print(f"\n[error] {event.get('error', 'unknown error')}")

    answer_text = "".join(answer_chunks)
    print("\n")
    _render_debug_sections(debug_payload)

    return {
        "query": query,
        "answer_text": answer_text,
        "debug": debug_payload,
        "events": events,
    }

## Run A Single Question

Edit `QUESTION` and run this cell.

In [3]:
QUESTION = "Customer's service if off for nonpayment.   How do I help the customer?"
result = ask_with_debug(QUESTION)

timing: search_embeddings 2.330s
timing: get_parent_extracts 0.003s
If the customer is **off for nonpayment**, help them based on the service type:

## Traditional (postpay) returning customer
- If they have a **write-off balance**:
  - If the write-off is **$25 or less** (or **over 4 years**) and they agree to pay it, you can **run a credit check** to determine whether a **deposit is required**.
  - If the write-off was **greater than $25**, their **credit score will NOT be restored** even after paying.
- If **deposit is waived**, it is **only good for 30 days**; after 30 days you must **run the credit check again** if service wasn’t started.
- If a **deposit is required**, the customer must pay **both the write-off and the deposit** before service can be started.
- When starting service, **review FAs**: if the last completed FA was **M-Off or M-cut for Non-Pay**, the **Start/Back-to-Back FA dispatch group must match** that last Non-Pay FA’s dispatch group.

## MyWay (Prepay) customer

## Question

Customer's service if off for nonpayment.   How do I help the customer?


## Query Rewrite

,field,value
0,status,not rewritten
1,reason,no_history
2,model,-
3,history_turns,-


...

,rank,score,from_vector,from_bm25,vector_score_norm,bm25_score_norm,vector_score_raw,bm25_score_raw,chunk_id,url
0,1,0.814832,True,True,0.885212,0.650610,0.691253,11.054746,2843,https://connections/?docs=residential%2Fstart-...
1,2,0.814832,True,True,0.885212,0.650610,0.691253,11.054746,1015,https://connections/?docs=residential/start-st...
2,3,0.790015,True,False,1.000000,0.300051,0.712239,5.542639,1235,https://connections/?docs=residential/myway/my...
3,4,0.790015,True,False,1.000000,0.300051,0.712239,5.542639,2349,https://connections/?docs=residential%2Fmyway%...
4,5,0.761855,True,False,0.953292,0.315171,0.703700,5.780390,2837,https://connections/?docs=residential%2Fstart-...
5,6,0.761855,True,False,0.953292,0.315171,0.703700,5.780390,722,https://connections/?docs=residential/start-st...
6,7,0.752522,True,False,0.960131,0.268102,0.704950,5.040280,423,https://connections/?docs=residential/outage-d...
7,8,0.752522,True,False,0.960131,0.268102,0.704950,5.040280,2535,https://connections/?docs=residential%2Foutage...
8,9,0.743857,True,False,0.931862,0.305178,0.699782,5.623260,477,https://connections/?docs=residential/billing-...
9,10,0.743857,True,False,0.931862,0.305178,0.699782,5.623260,1899,https://connections/?docs=residential%2Fbillin...


## LLM Request Payload

System:
Answer the question using only the provided context. If the answer is not in the context, say you don't know.

User (question + context):
Question: Customer's service if off for nonpayment.   How do I help the customer?

Context:
[glossary]
Glossary: 'Prepay' and 'MyWay' refer to the same program; treat them as identical terms.

---

[extract_id=716 score=0.8148]
Context: Home > Residential > Start, Stop & Transfer Traditional Service > Start Service (A) > Returning Customer – Write-Off Paid and Credit Restored

Once the customer agrees to pay the write-off balance that is $25.00 or less, or over 4 years, you can run a credit check to determine the deposit requirement.

__

##### IMPORTANT

If the Write-Off balance was greater than $25.00 you WILL NOT restore the credit score once the balance is paid.

# __Write-Off Paid & Deposit Waived

![](https://connections/wp-content/uploads/2021/10/writeoffpaid01.png)

__

Amount is $25 or Less

You can also review the Customer Contacts 

## LLM Response

If the customer is **off for nonpayment**, help them based on the service type:

## Traditional (postpay) returning customer
- If they have a **write-off balance**:
  - If the write-off is **$25 or less** (or **over 4 years**) and they agree to pay it, you can **run a credit check** to determine whether a **deposit is required**.
  - If the write-off was **greater than $25**, their **credit score will NOT be restored** even after paying.
- If **deposit is waived**, it is **only good for 30 days**; after 30 days you must **run the credit check again** if service wasn’t started.
- If a **deposit is required**, the customer must pay **both the write-off and the deposit** before service can be started.
- When starting service, **review FAs**: if the last completed FA was **M-Off or M-cut for Non-Pay**, the **Start/Back-to-Back FA dispatch group must match** that last Non-Pay FA’s dispatch group.

## MyWay (Prepay) customer
- If there’s an **active voucher**, review it in **MyUsage > Settin

## Optional Widget UI

Run this once to get a text box + button for repeated questions.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display

    question_box = widgets.Textarea(
        value="",
        placeholder="Type your question...",
        description="Question:",
        layout=widgets.Layout(width="100%", height="100px"),
    )
    top_k_slider = widgets.IntSlider(value=10, min=1, max=30, step=1, description="top_k")
    max_extracts_slider = widgets.IntSlider(value=6, min=1, max=12, step=1, description="extracts")
    ask_button = widgets.Button(description="Ask", button_style="primary")
    output = widgets.Output()

    def _on_click(_):
        with output:
            clear_output(wait=True)
            try:
                ask_with_debug(
                    question_box.value,
                    top_k=top_k_slider.value,
                    max_extracts=max_extracts_slider.value,
                )
            except Exception as exc:
                print(f"Failed: {exc}")

    ask_button.on_click(_on_click)
    display(widgets.VBox([question_box, widgets.HBox([top_k_slider, max_extracts_slider]), ask_button, output]))
except ImportError:
    print("ipywidgets is not installed. Use the single-question cell above instead.")